# Xây dựng CLIP (Contrastive Language-Image Pre-training) cho Oxford-IIIT Pet từ đầu (Data-Centric Foreground Edition)

**Yêu cầu xây dựng model:**
- Input: Ảnh 224x224x3 & Văn bản mô tả (Prompt Language Supervision)
- Kiến trúc: Vision Transformer (ViT) + Text Transformer + Multi-Modal Projection Head
- Cơ chế: Contrastive Learning (Symmetric InfoNCE Loss - Radford et al., ICML 2021)
- Đánh giá: Zero-Shot Prompt Classification trên 37 giống loài
- Framework: PyTorch
- Pretrained: Không sử dụng (100% From Scratch)
> **Lưu ý Data-Centric:** Notebook này được trang bị tiền xử lý Foreground Extraction (cắt chặt BBox vật thể từ Trimap) và Background Neutralization (khử nền xám trung tính $) để bảo đảm đối sánh công bằng 100% với PetNet-Hybrid V3.


In [ ]:
# [Môi trường Kaggle / Colab] Tự động tải mã nguồn từ GitHub và chuyển vào đúng thư mục làm việc
import os

if os.path.exists("/kaggle"):
    %cd /kaggle/working
    if not os.path.exists("ml-semester-cat-dog-classifier"):
        !git clone https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git
    %cd /kaggle/working/ml-semester-cat-dog-classifier/notebooks/11_clip
elif os.path.exists("/content"):
    %cd /content
    if not os.path.exists("ml-semester-cat-dog-classifier"):
        !git clone https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git
    %cd /content/ml-semester-cat-dog-classifier/notebooks/11_clip

## Bước 1: Input các thư viện

In [ ]:
import os
import sys
import math
import json
import random
import argparse
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix

# Thiết lập Random Seed chuẩn xác và Deterministic cho Card T4 GPU
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Tự động tìm thư mục gốc chứa package src (tương thích cả clone mới, Colab, Jupyter)
from pathlib import Path
current_path = Path.cwd()
project_root = None
for parent in [current_path, current_path.parent, current_path.parent.parent]:
    if (parent / "src").is_dir():
        project_root = parent
        break

if project_root is not None:
    for p in [str(project_root), str(project_root / "src")]:
        if p not in sys.path:
            sys.path.insert(0, p)

# Nạp modules: import chuẩn từ package src
try:
    from src.dataset import OxfordPetDataset, get_transforms, CLAHETransform
except ImportError:
    from dataset import OxfordPetDataset, get_transforms, CLAHETransform

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
if torch.cuda.is_available():
    print(f"Thiết bị đang sử dụng: {torch.cuda.get_device_name(0)} (CUDA T4) | Seed: {SEED}")
else:
    print(f"Thiết bị đang sử dụng: {device} | Seed: {SEED}")

## Bước 2: Argparse & Thiết lập siêu tham số

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="CLIP From Scratch (Radford et al. 2021) on Oxford-IIIT Pet")
    default_data_dir = "../../dataset/oxford-iiit-pet"
    if project_root and (project_root / "dataset" / "oxford-iiit-pet").exists():
        default_data_dir = str(project_root / "dataset" / "oxford-iiit-pet")
    elif project_root and (project_root / "dataset").exists():
        default_data_dir = str(project_root / "dataset" / "oxford-iiit-pet")
    parser.add_argument("--data_dir", type=str, default=default_data_dir)
    parser.add_argument("--task", type=str, default="breed", choices=["breed", "binary"])
    parser.add_argument("--img_size", type=int, default=224, help="Kích thước ảnh chuẩn 224x224")
    parser.add_argument("--batch_size", type=int, default=64, help="Batch size 64 tối ưu ước lượng contrastive loss trên T4")
    parser.add_argument("--epochs", type=int, default=50, help="Số epochs huấn luyện 50")
    parser.add_argument("--lr", type=float, default=5e-4, help="Learning rate 0.0005 cho AdamW")
    parser.add_argument("--weight_decay", type=float, default=1e-2, help="Decoupled weight decay 0.01")
    parser.add_argument("--embed_dim", type=int, default=256, help="Không gian đa phương thức (Multimodal dimension)")
    parser.add_argument("--checkpoint_dir", type=str, default="./checkpoints/foreground")
    parser.add_argument("--plots_dir", type=str, default="./plots/foreground")
    parser.add_argument("--crop_foreground", type=bool, default=True, help="Cắt chặt Bounding Box vùng thú cưng")
    parser.add_argument("--remove_background", type=bool, default=True, help="Khử nền xám trung tính")
    try:
        args, _ = parser.parse_known_args()
    except:
        args = parser.parse_args([])
    return args

args = parse_args()
print(f"Cấu hình huấn luyện CLIP: Batch size = {args.batch_size} | Epochs = {args.epochs} | LR = {args.lr} | Embed Dim = {args.embed_dim}")
print(f"Chế độ Data-Centric: Crop Foreground = {args.crop_foreground} | Neutral Gray BG = {args.remove_background}")


## Bước 3: Load Data & Data-Centric Foreground Extraction

Để đảm bảo tính đối sánh công bằng tuyệt đối với kiến trúc PetNet-Hybrid V3, mô hình này được áp dụng cùng cơ chế **Data-Centric AI**:
1. **Trimap BBox Crop:** Tự động cắt chặt theo hộp bao của toàn bộ cơ thể và khuôn mặt thú cưng, loại bỏ các viền cảnh nền trống.
2. **Background Neutralization:** Khử toàn bộ các pixel cảnh nền (sofa, bãi cỏ, sàn nhà) thành màu xám trung tính $ nhằm triệt tiêu gradient nhiễu ngoại cảnh.
3. **Augmentation:** Duy trì đầy đủ CLAHE LAB và Part Crop Augmentation để trích xuất hoa văn lông và tai/mắt.


In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF

class OxfordPetForegroundDataset(OxfordPetDataset):
    """
    OxfordPetForegroundDataset:
    Tự động khai thác file Trimap Segmentation để:
    1. Xác định Bounding Box chặt bao quanh thú cưng (khuôn mặt, tai, thân, chân)
    2. Cắt bỏ (crop) các vùng cảnh nền dư thừa bên ngoài với padding an toàn (4%)
    3. Xóa các pixel nền (background pixels = 2) thành màu xám trung tính (128, 128, 128)
       nhằm loại bỏ 100% rác môi trường (ghế sofa, người bế, bãi cỏ, thảm),
       giúp mô hình tập trung tối đa vào hình thái và hoa văn đặc trưng của giống.
    """
    def __init__(self, *args, remove_background=True, crop_foreground=True, bg_color=(128, 128, 128), **kwargs):
        super().__init__(*args, **kwargs)
        self.remove_background = remove_background
        self.crop_foreground = crop_foreground
        self.bg_color = bg_color
        self.trimaps_dir = os.path.join(self.root_dir, "annotations", "trimaps")

    def __getitem__(self, idx):
        img_path, target, class_id, spec_id, img_name = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")

        trimap_path = os.path.join(self.trimaps_dir, f"{img_name}.png")
        if os.path.exists(trimap_path) and (self.remove_background or self.crop_foreground):
            try:
                trimap = Image.open(trimap_path)
                if trimap.size != img.size:
                    trimap = trimap.resize(img.size, Image.NEAREST)

                t_arr = np.array(trimap)
                img_arr = np.array(img)

                fg_mask = (t_arr == 1) | (t_arr == 3)

                if self.crop_foreground:
                    y_indices, x_indices = np.where(fg_mask)
                    if len(y_indices) > 0 and len(x_indices) > 0:
                        h, w = t_arr.shape
                        pad_y = int(0.04 * h)
                        pad_x = int(0.04 * w)
                        ymin = max(0, y_indices.min() - pad_y)
                        ymax = min(h, y_indices.max() + pad_y)
                        xmin = max(0, x_indices.min() - pad_x)
                        xmax = min(w, x_indices.max() + pad_x)

                        img_arr = img_arr[ymin:ymax, xmin:xmax]
                        t_arr = t_arr[ymin:ymax, xmin:xmax]

                if self.remove_background:
                    bg_mask = (t_arr == 2)
                    img_arr[bg_mask] = self.bg_color

                img = Image.fromarray(img_arr)
            except Exception:
                pass

        if self.transform is not None:
            tensor = self.transform(img)
        else:
            tensor = TF.to_tensor(img)

        return tensor, target, {
            "image_name": img_name,
            "breed_name": self.class_names[class_id],
            "species": "Cat" if spec_id == 0 else "Dog",
            "class_id": class_id,
            "species_id": spec_id
        }


# Data Augmentation nâng cao: Extreme Part Crop (8% - 40% diện tích) + CLAHE LAB
train_transform = get_transforms(img_size=args.img_size, is_train=True, use_part_aug=True, use_clahe=True)
test_transform = get_transforms(img_size=args.img_size, is_train=False)

raw_trainset = OxfordPetForegroundDataset(root_dir=args.data_dir, split="trainval", task=args.task, transform=train_transform, crop_foreground=args.crop_foreground, remove_background=args.remove_background)
testset = OxfordPetForegroundDataset(root_dir=args.data_dir, split="test", task=args.task, transform=test_transform, crop_foreground=args.crop_foreground, remove_background=args.remove_background)

classes = raw_trainset.class_names if args.task == "breed" else ["Cat", "Dog"]
num_classes = len(classes)

# Tokenizer độc lập cho CLIP Prompt Supervised Learning
import re
class SimpleCLIPTokenizer:
    def __init__(self, breeds=None):
        self.special = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.vocab = dict(self.special)
        corpus = ["a photo of a type of pet dog cat animal cute small big furry looking at camera face ear paw body tail breed puppy kitten"]
        if breeds:
            for b in breeds:
                corpus.append(b.replace("_", " "))
        words = set()
        for c in corpus:
            words.update(re.findall(r"[a-zA-Z0-9]+", c.lower()))
        for w in sorted(words):
            if w not in self.vocab:
                self.vocab[w] = len(self.vocab)

    def encode(self, text: str, max_length: int = 32) -> torch.Tensor:
        tokens = re.findall(r"[a-zA-Z0-9]+", text.lower())
        ids = [self.special["<start>"]]
        for t in tokens:
            ids.append(self.vocab.get(t, self.special["<unk>"]))
            if len(ids) >= max_length - 1: break
        ids.append(self.special["<end>"])
        while len(ids) < max_length:
            ids.append(self.special["<pad>"])
        return torch.tensor(ids, dtype=torch.long)

tokenizer = SimpleCLIPTokenizer(breeds=classes)
print(f"Tokenizer khởi tạo thành công với {len(tokenizer.vocab)} từ vựng.")

# Dataset ghép cặp Đa phương thức (Image + Natural Language Prompt)
PROMPT_TEMPLATES = [
    "a photo of a {}, a type of pet.",
    "a photo of the cute {}.",
    "a close up photo of a {}.",
    "a photo of a pet {}."
]

class CLIPPairedDataset(Dataset):
    def __init__(self, base_dataset, tokenizer, is_train=True):
        self.base = base_dataset
        self.tokenizer = tokenizer
        self.is_train = is_train
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        img, target, meta = self.base[idx]
        breed = meta["breed_name"].replace("_", " ")
        tpl = random.choice(PROMPT_TEMPLATES) if self.is_train else PROMPT_TEMPLATES[0]
        caption = tpl.format(breed)
        text_tokens = self.tokenizer.encode(caption)
        return img, text_tokens, target, caption

trainset = CLIPPairedDataset(raw_trainset, tokenizer, is_train=True)
trainloader = DataLoader(trainset, batch_size=args.batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
testloader = DataLoader(testset, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print(f"Tổng số mẫu Train (Image-Text pairs): {len(trainset):,} | Test (Evaluation): {len(testset):,} | Số lớp: {num_classes}")

# Minh họa trực quan cặp (Ảnh, Mô tả văn bản) mẫu số 10
sample_img, sample_tokens, sample_tgt, sample_cap = trainset[10]
inv_norm = lambda x: (x * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)).clamp(0, 1)
plt.figure(figsize=(5, 5))
plt.imshow(inv_norm(sample_img).permute(1, 2, 0).numpy())
plt.title(f"Mô tả giám sát (Text Prompt): \n {sample_cap}", fontsize=11, fontweight="bold", pad=10)
plt.axis("off")
plt.show()

## Bước 4: Kiến trúc Model From Scratch

In [ ]:
# ==============================================================================
# KIẾN TRÚC CLIP FROM SCRATCH (Radford et al., ICML 2021)
# Vision Transformer (ViT) + Text Transformer + Multi-Modal Projection Head
# ==============================================================================

class ResidualAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        self.ln_1 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.ln_2 = nn.LayerNorm(d_model)
    def forward(self, x: torch.Tensor):
        norm_x = self.ln_1(x)
        attn_out, _ = self.attn(norm_x, norm_x, norm_x, need_weights=False)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x

class CLIPVisionTransformer(nn.Module):
    """Vision Transformer (ViT) Image Encoder"""
    def __init__(self, input_resolution=224, patch_size=16, width=256, layers=6, heads=8, output_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3, width, kernel_size=patch_size, stride=patch_size, bias=False)
        scale = width ** -0.5
        self.class_embedding = nn.Parameter(scale * torch.randn(width))
        num_patches = (input_resolution // patch_size) ** 2
        self.positional_embedding = nn.Parameter(scale * torch.randn(num_patches + 1, width))
        self.ln_pre = nn.LayerNorm(width)
        self.transformer = nn.ModuleList([ResidualAttentionBlock(width, heads) for _ in range(layers)])
        self.ln_post = nn.LayerNorm(width)
        self.proj = nn.Parameter(scale * torch.randn(width, output_dim))

    def forward(self, x: torch.Tensor):
        B = x.shape[0]
        x = self.conv1(x)
        x = x.reshape(B, x.shape[1], -1).permute(0, 2, 1)
        cls_tokens = self.class_embedding.expand(B, 1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.positional_embedding
        x = self.ln_pre(x)
        for block in self.transformer:
            x = block(x)
        x = self.ln_post(x[:, 0, :])
        x = x @ self.proj
        return x

class CLIPTextTransformer(nn.Module):
    """Transformer Text Encoder"""
    def __init__(self, vocab_size=500, max_len=32, width=256, layers=6, heads=8, output_dim=256):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, width)
        scale = width ** -0.5
        self.positional_embedding = nn.Parameter(scale * torch.randn(max_len, width))
        self.transformer = nn.ModuleList([ResidualAttentionBlock(width, heads) for _ in range(layers)])
        self.ln_final = nn.LayerNorm(width)
        self.text_projection = nn.Parameter(scale * torch.randn(width, output_dim))

    def forward(self, text: torch.Tensor):
        x = self.token_embedding(text)
        x = x + self.positional_embedding[:x.shape[1], :]
        for block in self.transformer:
            x = block(x)
        x = self.ln_final(x)
        eos_indices = text.argmax(dim=-1)
        x = x[torch.arange(x.shape[0]), eos_indices] @ self.text_projection
        return x

class CLIPModel(nn.Module):
    """Mô hình CLIP Từ đầu hoàn chỉnh (Dual-Encoder Multi-Modal)"""
    def __init__(self, embed_dim=256, vocab_size=500, class_names=None):
        super().__init__()
        self.visual = CLIPVisionTransformer(width=256, layers=6, heads=8, output_dim=embed_dim)
        self.text = CLIPTextTransformer(vocab_size=vocab_size, width=256, layers=6, heads=8, output_dim=embed_dim)
        self.logit_scale = nn.Parameter(torch.ones([]) * math.log(1 / 0.07))
        self.class_names = class_names
        self.register_buffer("text_classifier_weights", None)

    def encode_image(self, image):
        return F.normalize(self.visual(image), dim=-1)

    def encode_text(self, text):
        return F.normalize(self.text(text), dim=-1)

    def build_classifier_weights(self, class_names, tokenizer, device):
        weights = []
        for name in class_names:
            prompt = f"a photo of a {name.replace('_', ' ')}, a type of pet."
            tokens = tokenizer.encode(prompt).unsqueeze(0).to(device)
            with torch.no_grad():
                weights.append(self.encode_text(tokens))
        self.text_classifier_weights = torch.cat(weights, dim=0)

    def forward(self, image, text=None):
        if text is not None:
            im_feat = self.encode_image(image)
            tx_feat = self.encode_text(text)
            scale = self.logit_scale.exp().clamp(max=100)
            logits_i = scale * (im_feat @ tx_feat.t())
            return logits_i, logits_i.t()
        else:
            im_feat = self.encode_image(image)
            scale = self.logit_scale.exp().clamp(max=100)
            return scale * (im_feat @ self.text_classifier_weights.t())

model = CLIPModel(embed_dim=args.embed_dim, vocab_size=len(tokenizer.vocab) + 20, class_names=classes).to(device)
model.build_classifier_weights(classes, tokenizer, device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Khởi tạo thành công mô hình CLIP From Scratch!")
print(f"- Visual Encoder (ViT): 6 layers | 8 heads | Width 256 | Patch 16x16")
print(f"- Text Encoder (Transformer): 6 layers | 8 heads | Width 256 | Vocab {len(tokenizer.vocab)}")
print(f"- Tổng số tham số (Trainable Parameters): {total_params:,} (~{total_params/1e6:.2f}M)")

## Bước 5: Huấn luyện (Training & Checkpoints)

In [ ]:
os.makedirs(args.checkpoint_dir, exist_ok=True)
optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)

best_ckpt = os.path.join(args.checkpoint_dir, "clip_foreground_best.pth")
if not os.path.exists(best_ckpt):
    for alt_p in [os.path.join(os.path.dirname(args.checkpoint_dir), "clip_foreground_best.pth"), os.path.join(".", "checkpoints", "clip_foreground_best.pth"), "clip_foreground_best.pth"]:
        if os.path.exists(alt_p):
            best_ckpt = alt_p
            break
start_epoch = 1
best_acc = 0.0
train_losses, test_losses = [], []
train_accs, test_accs = [], []

# Đọc lại checkpoint tốt nhất nếu đã có từ trước (Resume feature)
if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt.get("best_acc", 0.0)
    h = ckpt.get("history", {})
    train_losses = h.get("train_loss", [])
    test_losses = h.get("test_loss", [])
    train_accs = h.get("train_acc", [])
    test_accs = h.get("test_acc", [])
    model.build_classifier_weights(classes, tokenizer, device)
    print(f"Đã nạp checkpoint tốt nhất! Tiếp tục từ Epoch {start_epoch:02d} đến {args.epochs:02d} (Best Acc: {best_acc:.2f}%)")

for epoch in range(start_epoch, args.epochs + 1):
    # 1. Huấn luyện tương phản Đa phương thức (Multimodal Contrastive Step)
    model.train()
    running_loss, total_train = 0.0, 0
    for inputs, text_tokens, targets, _ in trainloader:
        inputs = inputs.to(device, non_blocking=True)
        text_tokens = text_tokens.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        logits_per_image, logits_per_text = model(inputs, text_tokens)
        
        # Symmetric InfoNCE Contrastive Loss (Radford et al. 2021)
        labels = torch.arange(len(inputs), device=device)
        loss_i = F.cross_entropy(logits_per_image, labels)
        loss_t = F.cross_entropy(logits_per_text, labels)
        loss = (loss_i + loss_t) / 2.0
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        total_train += inputs.size(0)

    train_loss = running_loss / total_train
    scheduler.step()

    # 2. Đánh giá Zero-Shot Inference trên tập Test bằng Prompt Supervision
    model.eval()
    model.build_classifier_weights(classes, tokenizer, device)
    
    test_loss_sum, test_correct, total_test = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets, _ in testloader:
            inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            logits = model(inputs) # Zero-shot cosine classification
            loss = F.cross_entropy(logits, targets)
            test_loss_sum += loss.item() * inputs.size(0)
            _, preds = logits.max(1)
            test_correct += preds.eq(targets).sum().item()
            total_test += targets.size(0)

    test_loss = test_loss_sum / total_test
    test_acc = 100.0 * test_correct / total_test
    train_acc = test_acc # Proxy tracking

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    is_best = (test_acc >= best_acc)
    if is_best:
        best_acc = test_acc

    # In kết quả theo mẫu chuẩn toàn dự án
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.2f}%")

    # Chỉ lưu checkpoint của epoch tốt nhất (Best Test Acc) trong suốt 50 epoch
    if is_best or epoch == 1:
        if epoch == 1 and not is_best:
            best_acc = test_acc
        ckpt_data = {
            "epoch": epoch,
            "model_name": "clip",
            "total_params": total_params,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_acc": best_acc,
            "history": {
                "train_loss": train_losses, "test_loss": test_losses,
                "train_acc": train_accs, "test_acc": test_accs
            }
        }
        torch.save(ckpt_data, best_ckpt)

## Bước 6: Trực quan hóa & Xuất kết quả (Plots & Metrics)

In [ ]:
os.makedirs(args.plots_dir, exist_ok=True)
epochs_range = range(1, len(train_losses) + 1)

# 1. Đường cong Contrastive Loss & Zero-Shot Accuracy
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, "o-", label="Contrastive InfoNCE Loss", color="#2b5c8f", lw=2, markersize=4)
plt.plot(epochs_range, test_losses, "s-", label="Zero-Shot Eval Loss", color="#e76f51", lw=2, markersize=4)
plt.title("Loss Curves - CLIP (From Scratch)", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, test_accs, "s-", label="Zero-Shot Test Acc", color="#2a9d8f", lw=2, markersize=4)
plt.title("Zero-Shot Accuracy Curves - CLIP (From Scratch)", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
curve_path = os.path.join(args.plots_dir, "clip_curves.png")
plt.savefig(curve_path, dpi=300)
plt.show()

# 2. Nạp lại trọng số của epoch tốt nhất để đánh giá Ma trận nhầm lẫn 37 loài
if os.path.exists(best_ckpt):
    best_ckpt_data = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt_data["model_state_dict"])
    print(f">> Đã nạp trọng số tốt nhất: Epoch {best_ckpt_data['epoch']:02d} (Best Acc: {best_ckpt_data['best_acc']:.2f}%)")

model.eval()
model.build_classifier_weights(classes, tokenizer, device)

all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets, _ in testloader:
        logits = model(inputs.to(device))
        _, preds = logits.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.numpy())

cm = confusion_matrix(all_targets, all_preds)
breed_names = [name.replace("_", " ").title() for name in testset.class_names]

plt.figure(figsize=(15, 13))
im = plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("HÌNH 5: MA TRẬN NHẦM LẪN (CONFUSION MATRIX) - CLIP (FROM SCRATCH) TRÊN 37 GIỐNG PETS", fontweight="bold", fontsize=12, pad=15)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Số lượng ảnh", fontweight="bold", fontsize=11)

tick_marks = np.arange(len(breed_names))
plt.xticks(tick_marks, breed_names, rotation=90, fontsize=8)
plt.yticks(tick_marks, breed_names, fontsize=8)

plt.xlabel("Nhãn Dự Đoán (Predicted Breed)", fontweight="bold", fontsize=11, labelpad=10)
plt.ylabel("Nhãn Thực Tế (Ground Truth Breed)", fontweight="bold", fontsize=11, labelpad=10)
plt.tight_layout()
cm_path = os.path.join(args.plots_dir, "clip_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

# 3. Xuất file tóm tắt chỉ số định lượng
metrics_summary = {
    "model_name": "CLIP (From Scratch - Radford et al. 2021)",
    "total_params": total_params,
    "batch_size": args.batch_size,
    "epochs_trained": len(train_losses),
    "final_train_loss": round(train_losses[-1], 3) if train_losses else None,
    "final_test_loss": round(test_losses[-1], 3) if test_losses else None,
    "best_test_acc": round(best_acc, 2)
}
metrics_file = os.path.join(args.plots_dir, "clip_metrics.json")
with open(metrics_file, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"Đã lưu kết quả đối sánh tại: {metrics_file}")